# -*- coding: utf-8 -*-
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:  Luke Lozenski (DTU)

# Linearized/Proximal ADMM varying and adaptive penalty paramter demo

The adaptive method of multipliers (ADMM) is method of solving constrained optimization problem utilizing proximals. CIL has an implementation for the proximal form for solving idenity constrained problems, but this can in general be implemented for all linear equality and convex inequality constrined optimization problems. These optimization problems can be expressed in uncstrained for as 

$$ \min_x f(x) + g(Kx),$$

which is equivalent to the constrained form

$$\begin{array}{cc}
   \min_{x,z} & f(x) + g(z) \\
   \textnormal{s.t.} & Kx - z = 0
\end{array}.$$

Linearized ADMM, which is also reffered to as proximal ADMM, solves this optimization problem via the iterative process involving a Lagrangian multiplier variable $u$

$$x^{k+1} = \operatorname{prox}_{f,1/(\mu \rho)}\left(x^k - \frac{1}{\mu} K^T(Kx^k - z^k + \frac{u^k}{\rho}) \right)$$

$$z^{k+1} = \operatorname{prox}_{g,1/(\nu \rho)}\left(z^k - \frac{1}{\nu}(Kx^{k+1} - z^k + \frac{u^k}{\rho}) \right)$$
$$u^{k+1} = u^k + \rho(Kx^{k+1} - z^{k+1}),$$

where $\mu$ and $\nu$ are chosen such that $\mu > \| K\|^2_{op}$ and $\nu > 1$. CIL had previosly only implemented the $\rho = 1$ case, which will have different convergence speeds and behaviours dependent on scaling. I added the ability to chose $\rho$ to any user specific value with the default value still being 1. I additionally added a functionality for an adaptive penality parameter $\rho^k$ that choses adaptively chooses optimal penalty parameter for accelerated convergence via the rule 

$$\left(\rho^{k+1}\right)^2 =  \frac{3 \| u^{(k+1)} - u^{(k)}\|^2 + 2{\rho^{(k)}}^2 
 (\nu - 1)\|z^{(k+1)} - 2z^{(k)} +  z^{(k-1)}\|^2 + {\rho^{(k)}}^2 \|x^{(k+1)} -2x^{(k)} +  x^{(k-1)}\|^2_{\mu I - K^T K}}{ (2\nu - 1) \| z^{(k+1)} - z^{(k)}\|^2 + \| x^{(k+1)} - x^{(k)}\|^2_{\mu I - K^TK} }$$

Here I demonstrate how to implemente this for L2 data fidelity problems with Tikhohonov and Total variation regularized problems

First we generate a ground truth object with the Shepp-Logan phantom, generate its sinogram, and add noise to it

In [22]:
import cil
from cil.utilities.display import show2D
from cil.plugins import TomoPhantom as cilTomoPhantom
from cil.framework import  AcquisitionGeometry, ImageData, AcquisitionData, ImageGeometry, BlockDataContainer
from cil.plugins.astra.operators import ProjectionOperator

n = 512
ig = ImageGeometry(voxel_num_x=n, 
                    voxel_num_y=n, 
                    voxel_size_x=1, 
                    voxel_size_y=1)
X = cilTomoPhantom.get_ImageData(num_model=1, geometry=ig)

show2D(X)



In [23]:
import numpy as np
n_angles = int(np.pi*n)
angles = np.linspace(0,180,n_angles)

ag = AcquisitionGeometry.create_Parallel2D().set_angles(angles).set_panel(n, pixel_size=1.0)
ig = ag.get_ImageGeometry()

A =  ProjectionOperator(ig, ag, device='gpu')
Y = A.direct(X)

Ynoisy = Y.copy()
snr = 0.1
std = snr*np.mean(Y.array**2)**(1/2)
Ynoisy.array += std*np.random.standard_normal(Y.array.shape)


show2D([Y,Ynoisy])

The Tihonov regularized reconstruction can be formulated as an unconstrained problem

$$\min_x \|A x- d\|^2 + \alpha\| \nabla x\|^2 = \| \begin{pmatrix} A \\ \sqrt{\alpha} \nabla \end{pmatrix} x - \begin{pmatrix} d \\ 0 \end{pmatrix}\|^2,$$

where the global optimum can be found using CGLS and used to compute the convergence of other methods

In [24]:
from cil.optimisation.algorithms import CGLS
from cil.optimisation.functions import L2NormSquared, L1Norm, BlockFunction, MixedL21Norm, IndicatorBox, TotalVariation, ZeroFunction
from cil.optimisation.operators import GradientOperator, BlockOperator
import matplotlib.pyplot as plt
alpha = 1e5
Grad = np.sqrt(alpha)*GradientOperator(ig)
K = BlockOperator(Grad, A)
Y_block = BlockDataContainer( Grad.range.allocate(0), Ynoisy)

x0 = ig.allocate(0)
cgls = CGLS(initial=x0, operator=K, data=Y_block)
cgls.run(100, verbose=1)

show2D(cgls.solution, title="CGLS")
plt.semilogy(cgls.objective)
plt.title('Relative Root Mean Square Error')




The unconstrained problem is equivalent to the optimized form 


$$\begin{array}{cc} \min_{x,z} &\|z - \begin{pmatrix} d \\ 0 \end{pmatrix}\|^2  \\ s.t. &  \begin{pmatrix} A \\ \sqrt{\alpha} \nabla \end{pmatrix} x - z = 0\end{array},$$

which can be solved using Linearized ADMM and Primal Dual Hybrid Gradient (PDHG).

In [25]:
from cil.optimisation.algorithms import PDHG

cgls_norm = np.mean(cgls.solution.array**2)**(1/2)
pdhg_errs = []
num_iters = 200


Grad = np.sqrt(alpha)*GradientOperator(ig)
beta = 1/10
K = BlockOperator(Grad, A)*beta
f1 = L2NormSquared()
f2 = L2NormSquared(b=Ynoisy*beta)
F = BlockFunction(f1, f2)
G = ZeroFunction()
pdhg = PDHG(f = F, g = G, operator = K,
            update_objective_interval = 1, check_convergence=False)

for iter in range(num_iters):
    pdhg.run(1, verbose=0)
    err = np.mean( (pdhg.solution.array - cgls.solution.array)**2)**(1/2)
    pdhg_errs.append(err/cgls_norm)

show2D(pdhg.solution, title = "PDHG")

In [26]:
from cil.optimisation.algorithms import LADMM

admm_errs = []

admm = LADMM(f = G, g = F, operator = K)

for iter in range(num_iters):
    admm.run(1, verbose=0)
    err = np.mean( (admm.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs.append(err/cgls_norm)

show2D(admm.solution, title = "ADMM")
plt.semilogy(pdhg_errs, label = 'pdhg')
plt.semilogy(admm_errs, label = 'admm')
plt.legend()
plt.title('Relative Root Mean Square Error')



Users can now manually choose the value of the penalty parameter $\rho$, which impacts convergence speed.

In [27]:
admm_1 = LADMM(f = G, g = F, operator = K, rho = 1e1)
admm_errs_1 = []
admm_n1 = LADMM(f = G, g = F, operator = K, rho = 1e-1)
admm_errs_n1 = []

for iter in range(num_iters):
    admm_1.run(1, verbose=0)
    admm_n1.run(1, verbose=0)

    err = np.mean( (admm_1.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_1.append(err/cgls_norm)

    err = np.mean( (admm_n1.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_n1.append(err/cgls_norm)

show2D([admm_1.solution, admm_n1.solution], title= [r"$\rho = 10^1$", r"$\rho = 10^{-1}$"])


In [28]:
fig, a = plt.subplots(1,2, figsize = (10,5))
a[0].semilogy(pdhg_errs, label = 'pdhg')
a[0].semilogy(admm_errs, label = r'admm $\rho = 1$')
a[0].semilogy(admm_errs_1, label = r'admm $\rho = 10^1$')
a[0].semilogy(admm_errs_n1, label = r'admm $\rho = 10^{-1}$')



a[1].semilogy(pdhg.objective, label = 'pdhg')
a[1].semilogy(admm.objective, label = r'admm $\rho = 1$')
a[1].semilogy(admm_1.objective, label = r'admm $\rho = 10^1$')
a[1].semilogy(admm_n1.objective, label = r'admm $\rho = 10^{-1}$')
a[0].set_title('Relative Root Mean Square Error')
a[1].set_title('Objective Value')
a[1].legend()


The penalty parameter $\rho$ can also be set using the adaptive rule which finds the optimum weighting regardless of the initial value $\rho^0$, which the user can set.

In [29]:
admm_ad = LADMM(f = G, g = F, operator = K, mode = 'adaptive')
admm_errs_ad = []
for iter in range(num_iters):
    admm_ad.run(1, verbose=0)
    err = np.mean( (admm_ad.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_ad.append(err/cgls_norm)
show2D(admm_ad.solution)

In [30]:
fig, a = plt.subplots(1,2, figsize = (10,5))
a[0].semilogy(pdhg_errs, label = 'pdhg')
a[0].semilogy(admm_errs, label = r'admm $\rho = 1$')
a[0].semilogy(admm_errs_1, label = r'admm $\rho = 10^1$')
a[0].semilogy(admm_errs_n1, label = r'admm $\rho = 10^{-1}$')
a[0].semilogy(admm_errs_ad, label = r'admm adaptive $\rho$')

a[1].semilogy(pdhg.objective, label = 'pdhg')
a[1].semilogy(admm.objective, label = r'admm $\rho = 1$')
a[1].semilogy(admm_1.objective, label = r'admm $\rho = 10^1$')
a[1].semilogy(admm_n1.objective, label = r'admm $\rho = 10^{-1}$')
a[1].semilogy(admm_ad.objective, label = r'admm adaptive $\rho$')
a[0].set_title('Relative Root Mean Square Error')
a[1].set_title('Objective Value')
a[1].legend()

In [31]:
admm_0 = LADMM(f = G, g = F, operator = K, rho = 1e0, mode = 'adaptive')
admm_errs_0 = []

admm_1 = LADMM(f = G, g = F, operator = K, rho = 1e1, mode = 'adaptive')
admm_errs_1 = []
admm_n1 = LADMM(f = G, g = F, operator = K, rho = 1e-1, mode = 'adaptive')
admm_errs_n1 = []

admm_2 = LADMM(f = G, g = F, operator = K, rho = 1e2,  mode = 'adaptive')
admm_errs_2 = []
admm_n2 = LADMM(f = G, g = F, operator = K, rho = 1e-2, mode = 'adaptive')
admm_errs_n2 = []

for iter in range(num_iters):
    admm_0.run(1, verbose=0)

    admm_1.run(1, verbose=0)
    admm_n1.run(1, verbose=0)

    admm_2.run(1, verbose=0)
    admm_n2.run(1, verbose=0)

    err = np.mean( (admm_0.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_0.append(err/cgls_norm)

    err = np.mean( (admm_1.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_1.append(err/cgls_norm)

    err = np.mean( (admm_n1.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_n1.append(err/cgls_norm)

    err = np.mean( (admm_2.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_2.append(err/cgls_norm)

    err = np.mean( (admm_n2.solution.array - cgls.solution.array)**2)**(1/2)
    admm_errs_n2.append(err/cgls_norm)


In [32]:
fig, a = plt.subplots(1,2, figsize = (10,5))
a[0].semilogy(pdhg_errs, label = 'pdhg')
a[0].semilogy(admm_errs_n2, label = r'admm $\rho^0 = 10^{-2}$')
a[0].semilogy(admm_errs_n1, label = r'admm $\rho^0 = 10^{-1}$')
a[0].semilogy(admm_errs_0, label = r'admm $\rho^0 = 1$')
a[0].semilogy(admm_errs_1, label = r'admm $\rho^0 = 10^{1}$')
a[0].semilogy(admm_errs_2, label = r'admm $\rho^0 = 10^{2}$')

a[1].semilogy(pdhg.objective, label = 'pdhg')
a[1].semilogy(admm_n2.objective, label = r'admm $\rho^0 = 10^{-2}$')
a[1].semilogy(admm_n1.objective, label = r'admm $\rho^0 = 10^{-1}$')
a[1].semilogy(admm_0.objective, label = r'admm $\rho^0 = 1$')
a[1].semilogy(admm_1.objective, label = r'admm $\rho^0 = 10^{1}$')
a[1].semilogy(admm_2.objective, label = r'admm $\rho^0 = 10^{2}$')
a[0].set_title('Relative Root Mean Square Error')
a[1].set_title('Objective Value')
a[1].legend()


We can similiarly apply Linearized ADMM and PDHG to solve an $L^2$ data fidelity with TV regulariztion. This corresponds to the unconstrained problem

$$\min_x \|Ax - d\|^2 + \alpha\| \nabla x\|_{2,1}$$


The unconstrained problem is equivalent to the optimized form 


$$\begin{array}{cl} \min_{x,z} &\|z_1 - d\|^2 + \|z_2\|_{2,1}  \\ s.t. &  \begin{pmatrix} A \\ \alpha \nabla \end{pmatrix} x - \begin{pmatrix}z_1 \\ z_2 \end{pmatrix}=0\end{array},$$

which can be solved using Linearized ADMM and PDHG. Here we see that Linearized ADMM with the adaptive penalty parameter performs much better than PDHG

In [33]:
alpha = 1e3
Grad = alpha*GradientOperator(ig)
K = BlockOperator(Grad, A)
f1 = MixedL21Norm()
f2 = L2NormSquared(b=Ynoisy)
F = BlockFunction(f1, f2)
G = ZeroFunction()
num_iters = 200
pdhg = PDHG(f = F, g = G, operator = K,
            update_objective_interval = 1, check_convergence=False)
pdhg.run(num_iters)
admm_0 = LADMM(f = G, g = F, operator = K, rho = 1e0, mode = 'adaptive')
admm_0.run(num_iters)
admm_1 = LADMM(f = G, g = F, operator = K, rho = 1e1, mode = 'adaptive')
admm_1.run(num_iters)
admm_n1 = LADMM(f = G, g = F, operator = K, rho = 1e-1, mode = 'adaptive')
admm_n1.run(num_iters)
admm_2 = LADMM(f = G, g = F, operator = K, rho = 1e2,  mode = 'adaptive')
admm_2.run(num_iters)
admm_n2 = LADMM(f = G, g = F, operator = K, rho = 1e-2, mode = 'adaptive')
admm_n2.run(num_iters)

In [34]:
plt.semilogy(pdhg.objective, label = 'pdhg')
plt.semilogy(admm_n2.objective, label = r'admm $\rho^0 = 10^{-2}$')
plt.semilogy(admm_n1.objective, label = r'admm $\rho^0 = 10^{-1}$')
plt.semilogy(admm_0.objective, label = r'admm $\rho^0 = 1$')
plt.semilogy(admm_1.objective, label = r'admm $\rho^0 = 10^{1}$')
plt.semilogy(admm_2.objective, label = r'admm $\rho^0 = 10^{2}$')
plt.legend()

plt.title('Objective Value')
plt.show()

show2D([pdhg.solution, admm_0.solution], title=["PDHG", "ADMM"])

Reimplementing everything in the equivalent constrained form leads to different convergence behavior for both Linearized ADMM and PDHG

$$\begin{array}{cl} \min_{x,z} &\|z_1 - d\|^2 + \alpha\|z_2\|_{2,1}  \\ s.t. &  \begin{pmatrix} A \\  \nabla \end{pmatrix} x - \begin{pmatrix}z_1 \\ z_2 \end{pmatrix}=0\end{array}.$$

This can be remedied in the future with multiparameter methods. 

In [35]:
alpha = 1e3
Grad = GradientOperator(ig)
K = BlockOperator(Grad, A)
f1 = alpha*MixedL21Norm()
f2 = L2NormSquared(b=Ynoisy)
F = BlockFunction(f1, f2)
G = ZeroFunction()
num_iters = 45
pdhg = PDHG(f = F, g = G, operator = K,
            update_objective_interval = 1, check_convergence=False)
pdhg.run(num_iters)
admm_0 = LADMM(f = G, g = F, operator = K, rho = 1e0, mode = 'adaptive')
admm_0.run(num_iters)
admm_1 = LADMM(f = G, g = F, operator = K, rho = 1e1, mode = 'adaptive')
admm_1.run(num_iters)
admm_n1 = LADMM(f = G, g = F, operator = K, rho = 1e-1, mode = 'adaptive')
admm_n1.run(num_iters)
admm_2 = LADMM(f = G, g = F, operator = K, rho = 1e2,  mode = 'adaptive')
admm_2.run(num_iters)
admm_n2 = LADMM(f = G, g = F, operator = K, rho = 1e-2, mode = 'adaptive')
admm_n2.run(num_iters)





In [36]:
plt.semilogy(pdhg.objective, label = 'pdhg')
plt.semilogy(admm_n2.objective, label = r'admm $\rho^0 = 10^{-2}$')
plt.semilogy(admm_n1.objective, label = r'admm $\rho^0 = 10^{-1}$')
plt.semilogy(admm_0.objective, label = r'admm $\rho^0 = 1$')
plt.semilogy(admm_1.objective, label = r'admm $\rho^0 = 10^{1}$')
plt.semilogy(admm_2.objective, label = r'admm $\rho^0 = 10^{2}$')
plt.title('Objective Value')
plt.legend()
plt.show()


show2D([pdhg.solution, admm_0.solution], title=["PDHG", "ADMM"])

In [37]:

num_iters = 200
pdhg = PDHG(f = F, g = G, operator = K,
            update_objective_interval = 1, check_convergence=False)
pdhg.run(num_iters)
admm_0 = LADMM(f = G, g = F, operator = K, rho = 1e0, mode = 'adaptive')
admm_0.run(num_iters)
admm_1 = LADMM(f = G, g = F, operator = K, rho = 1e1, mode = 'adaptive')
admm_1.run(num_iters)
admm_n1 = LADMM(f = G, g = F, operator = K, rho = 1e-1, mode = 'adaptive')
admm_n1.run(num_iters)
admm_2 = LADMM(f = G, g = F, operator = K, rho = 1e2,  mode = 'adaptive')
admm_2.run(num_iters)
admm_n2 = LADMM(f = G, g = F, operator = K, rho = 1e-2, mode = 'adaptive')
admm_n2.run(num_iters)

plt.semilogy(pdhg.objective, label = 'pdhg')
plt.semilogy(admm_n2.objective, label = r'admm $\rho^0 = 10^{-2}$')
plt.semilogy(admm_n1.objective, label = r'admm $\rho^0 = 10^{-1}$')
plt.semilogy(admm_0.objective, label = r'admm $\rho^0 = 1$')
plt.semilogy(admm_1.objective, label = r'admm $\rho^0 = 10^{1}$')
plt.semilogy(admm_2.objective, label = r'admm $\rho^0 = 10^{2}$')
plt.title('Objective Value')
plt.legend()
plt.show()

show2D([pdhg.solution, admm_0.solution], title=["PDHG", "ADMM"])
